In [2]:
import wandb
wandb.login()

wandb: Currently logged in as: baymaxnguyen306 (baymaxnguyen306-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import tqdm

In [4]:
project ="First testing project"
config={"epoch":5,
        'lr':0.01}
with wandb.init(project=project, config= config) as run:
    run.log({"accuracy":0.9, "loss":0.1})

accuracy,▁
loss,▁
accuracy,0.9
loss,0.1


In [5]:
class ConvNet(nn.Module):
    def __init__(self,para):
        super(ConvNet, self).__init__()
        self.para=para
        self.num_classes=para['num_classes']
        self.num_CNN_layer=para['num_CNN_layer']
        self.num_FC_layer=para['num_FC_layer']
        self.kernel_size=para['kernel_size']
        self.CNN_channels=para['CNN_channels']
        self.FC_neurons=para['FC_neurons']
        self.input_channel=para['input_size']
        self.input_image_size=para['input_image_size']
        self.Conv_layer=nn.ModuleList()
        self.fc_layer=nn.ModuleList()
        self.classifier=nn.Linear(self.FC_neurons[-1], self.num_classes)
        self.Conv_layers_build()
        self.fc_layer_build()

    def Conv_layers_build(self):
        for i in range(self.num_CNN_layer):
            input_channel=1 if i==0 else self.CNN_channels[i-1]
            output_channel=self.CNN_channels[i]
            Conv_layer=nn.Sequential(
                nn.Conv2d(input_channel,output_channel, kernel_size=self.kernel_size[i],stride=1, padding=1),
                nn.BatchNorm2d(output_channel),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )
            self.Conv_layer.append(Conv_layer)
    def conv_forward(self,x):
        for layer in self.Conv_layer:
            x=layer(x)
        return x
    def fc_layer_build(self):
        in_features=0
        with torch.no_grad():
            dummy_input=torch.zeros(1, self.input_channel, self.input_image_size[0], self.input_image_size[1])
            out=self.conv_forward(dummy_input)
            in_features=out.flatten(1).shape[1]
        for i in range(self.num_FC_layer):
            input_features=in_features if i==0 else self.FC_neurons[i-1]
            output_features=self.FC_neurons[i]
            fc_layer=nn.Sequential(
                nn.Linear(input_features, output_features),
                nn.ReLU()
            )
            self.fc_layer.append(fc_layer)
    def fc_forward(self,x):
        x=x.flatten(1)
        for layer in self.fc_layer:
            x=layer(x)
        return x
    
    def forward(self, x):
        x=self.conv_forward(x)
        x=self.fc_forward(x)
        x=self.classifier(x)
        out=torch.softmax(x, dim=1)
        return out
    def summary(self):
        print("="*40)
        print(f"{'Layer':<15}{'Output Shape':<20}{'Details'}")
        print("="*40)
        for i, conv in enumerate(self.Conv_layer):
            print(f"Conv{i+1:<10} {str(conv):<20}")
        for i, fc in enumerate(self.fc_layer):
            print(f"FC{i+1:<10} {str(fc):<20}")
        print("="*40)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Total Parameters: {total_params}")
if __name__ == "__main__":
    para={
        'num_classes':10,
        'num_CNN_layer':2,
        'num_FC_layer':2,
        'kernel_size':[3,3],
        'CNN_channels':[16,32],
        'FC_neurons':[128,64],
        'input_size':1,
        'input_image_size':[620,620]
    }
    model=ConvNet(para)
    model.summary()
    x=torch.randn(64,1,620,620)
    out=model(x)
    print(out.shape)

Layer          Output Shape        Details
Conv1          Sequential(
  (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
Conv2          Sequential(
  (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
FC1          Sequential(
  (0): Linear(in_features=768800, out_features=128, bias=True)
  (1): ReLU()
)
FC2          Sequential(
  (0): Linear(in_features=128, out_features=64, bias=True)
  (1): ReLU()
)
Total Parameters: 98420330
torch.Size([64, 10])


In [6]:
def get_data(slice,Train=True):
    full_dataset=torchvision.datasets.MNIST(root='./data', train=Train, download=True, transform=transforms.ToTensor())
    sub_set=torch.utils.data.Subset(full_dataset, indices=range(0,len(full_dataset),slice))
    return sub_set



In [11]:
class TrainerWB:
    def __init__(self,config,train_set,test_set,project_name):
        self.config=config
        self.learning_rate=config['lr']
        self.epochs=config['epoch']
        self.batch_size=config['batch_size']
        self.device='cuda'if torch.cuda.is_available() else 'cpu'
        self.train_set=train_set
        self.test_set=test_set
        self.train_loader=DataLoader(train_set, batch_size=self.batch_size, shuffle=True)
        self.test_loader=DataLoader(test_set, batch_size=self.batch_size, shuffle=False)
        self.model=ConvNet(self.config['model_para']).to(self.device)
        self.criterion=nn.CrossEntropyLoss()
        self.optimizer=self.optim_build()
        self.project_name=project_name
    def optim_build(self):
        if self.config['optim']=='SGD':
            return optim.SGD(self.model.parameters(), lr=self.learning_rate, momentum=0.9)
        elif self.config['optim']=='Adam':
            return optim.Adam(self.model.parameters(), lr=self.learning_rate)
        else:
            raise ValueError("Unsupported optimizer type")
    def train(self):
        self.model.train()
        running_loss=0.0 
        correct=9
        total=0
        loop=tqdm.tqdm(self.train_loader,desc='Tao dang training',leave=False)
        for image, label in loop:
            image,label= image.to(self.device), label.to(self.device)
            #Forward propagation:
            output=self.model(image)
            loss=self.criterion(output,label)
            #backward propagation
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            #Metrics calculation
            running_loss += loss.item()
            preds=torch.argmax(output,dim=1)
            correct += (preds==label).sum().item()
            total += label.size(0)
            #Update progress bar
            loop.set_postfix(loss=running_loss/ (total/self.batch_size), accuracy=100.* correct/ total)
        avg_loss = running_loss / total
        accuracy = correct / total
        return avg_loss, accuracy
        
    def validate(self):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in self.test_loader:
                images, labels = images.to(self.device), labels.to(self.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                
        avg_loss = running_loss / total
        accuracy = correct / total
        return avg_loss, accuracy

    def fit(self):
        with wandb.init(project=self.project_name, config=self.config) as run:
            wandb.watch(self.model, log="all",log_freq=10)
            print(f"Starting training on {self.device}...")
            for epoch in range(self.epochs):
                train_loss, train_acc = self.train()
                val_loss, val_acc = self.validate()
                
                print(f"Epoch [{epoch+1}/{self.epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
                
                wandb.log({
                    "Train Loss": train_loss,
                    "Train Accuracy": train_acc,
                    "Validation Loss": val_loss,
                    "Validation Accuracy": val_acc
                })
        print("Training complete.")
    

In [15]:
def main():
    config={
        'lr':0.001,
        'epoch':5,
        'batch_size':64,
        'optim':'SGD',
        'model_para':{
            'num_classes':10,
            'num_CNN_layer':2,
            'num_FC_layer':2,
            'kernel_size':[3,3],
            'CNN_channels':[16,32],
            'FC_neurons':[128,64],
            'input_size':1,
            'input_image_size':[28,28]
        }
    }
    train_set=get_data(slice=6, Train=True)
    test_set=get_data(slice=6, Train=False)
    trainer=TrainerWB(config, train_set, test_set, project_name="MNIST_Classification_WandB")
    trainer.fit()


In [19]:
sweep_config={
    'method':'random',
    'metric':{
        'name':'Validation Accuracy',
        'goal':'maximize'
    },
    'parameters':{
        'lr':{
            'values':[0.01,0.001,0.0001]
        },
        'epoch':{
            'values':[5,6,7]
        },
        'batch_size':{
            'values':[16,32,64]
        },
        'optim':{
            'values':['SGD','Adam']
        },
        'model_para':{
            'values':[{
                'num_classes':10,
                'num_CNN_layer':2,
                'num_FC_layer':2,
                'kernel_size':[3,3],
                'CNN_channels':[16,32],
                'FC_neurons':[128,64],
                'input_size':1,
                'input_image_size':[28,28]
            },
            {
                'num_classes':10,
                'num_CNN_layer':3,
                'num_FC_layer':2,
                'kernel_size':[3,3,3],
                'CNN_channels':[16,32,64],
                'FC_neurons':[256,128],
                'input_size':1,
                'input_image_size':[28,28]
            }]
        
        }
    }
}

In [23]:
sweep_id= wandb.sweep(sweep_config, project="MNIST_Classification_WandB")

Create sweep with ID: 5takkkev
Sweep URL: https://wandb.ai/baymaxnguyen306-ho-chi-minh-city-university-of-technology/MNIST_Classification_WandB/sweeps/5takkkev


In [24]:
wandb.agent(sweep_id, function=main, count=10)

wandb: Agent Starting Run: trflrhia with config:
wandb: 	batch_size: 64
wandb: 	epoch: 5
wandb: 	lr: 0.0001
wandb: 	model_para: {'CNN_channels': [16, 32], 'FC_neurons': [128, 64], 'input_image_size': [28, 28], 'input_size': 1, 'kernel_size': [3, 3], 'num_CNN_layer': 2, 'num_FC_layer': 2, 'num_classes': 10}
wandb: 	optim: Adam


Starting training on cuda...


Epoch [1/5], Train Loss: 0.0361, Train Acc: 0.1453, Val Loss: 2.2937, Val Acc: 0.2352


Epoch [2/5], Train Loss: 0.0359, Train Acc: 0.2659, Val Loss: 2.2758, Val Acc: 0.2603


Epoch [3/5], Train Loss: 0.0354, Train Acc: 0.3139, Val Loss: 2.2313, Val Acc: 0.3395


...

Epoch [3/5], Train Loss: 0.0356, Train Acc: 0.4032, Val Loss: 2.2432, Val Acc: 0.4217


Epoch [4/5], Train Loss: 0.0345, Train Acc: 0.3965, Val Loss: 2.1462, Val Acc: 0.4187


Epoch [5/5], Train Loss: 0.0326, Train Acc: 0.4849, Val Loss: 2.0135, Val Acc: 0.5471


Train Accuracy,▁▅▆▆█
Train Loss,██▇▅▁
Validation Accuracy,▁▄▄▄█
Validation Loss,██▇▄▁
Train Accuracy,0.4849
Train Loss,0.03259
Validation Accuracy,0.54709
Validation Loss,2.01347


Training complete.
